In [6]:
import feedparser
import arxiv
from datetime import datetime, timedelta
import urllib.parse

def get_google_news(keywords, days=7):
    """
    구글 뉴스 RSS를 통해 키워드 관련 뉴스를 가져옵니다.
    """
    base_url = "https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko"
    # 최근 n일간의 검색 결과 필터링 (when:7d 방식)
    query = f"{keywords} when:{days}d"
    encoded_query = urllib.parse.quote(query)
    rss_url = base_url.format(query=encoded_query)
    
    feed = feedparser.parse(rss_url)
    results = []
    
    for entry in feed.entries:
        results.append({
            "title": entry.title,
            "link": entry.link,
            "published": entry.published,
            "source": "Google News"
        })
    return results

In [11]:
news = get_google_news("난임", days=7)

In [13]:
for n in news[:10]:
    print(f"[{n['published']}] {n['title']}\nURL: {n['link']}\n")

[Wed, 25 Feb 2026 19:30:00 GMT] 엄마 혈장으로 배아 키운다… ‘고령 난임’ 임신율 1.8배 올라 - 동아일보
URL: https://news.google.com/rss/articles/CBMib0FVX3lxTE5yMHhxX1RGNEkzUlZqNUlMaXNPRWlQdU1jTlVZV2c0QlNFRFBUSkFwLTkzaDdyZTI0Q1lDRmtVN3hUeGZDanpSaTNLdTVvNkZHTHM3ekhlbklGU1F5U0tFYTYxNzEzREtmTkZQYTUxc9IBZkFVX3lxTE9WSVVBTWFZV1RFM1lHNHhweWRxX0NJazFadzctSm9KUThGdHNtc3BSaWZiclJLWmRfblVlbTNlMGNVMTJPRlNuNEo3ZFdfTkRXSGhwRVdSc293aXpLclNIV2lqSWNDZw?oc=5

[Tue, 24 Feb 2026 07:01:34 GMT] 인구보건복지협회, 사실혼 포함 난임 부부에 최대 50만원 지원 - 연합뉴스
URL: https://news.google.com/rss/articles/CBMiW0FVX3lxTE1hZDJKTUQ1TVdfb0V0XzVpcTl6UVJYWFBDeTVoYUxSRk9Ob0o4M21wWXFEX3owME9NZ3Yxc193TTNoLXJUaEpfal9ROVFMRTl6T3pabHd1eUtvWDjSAWBBVV95cUxQUGN6TGdNLU1uXzZtSGloLTR0eXNCbER2QXRCd0R5NU96Wk51OURfWE1TVDUycHg3bUlvRTMtSUZvcG9nc2EtY3pGcXdEanpJR0VHaVBsYS1veWRiaEF1U3E?oc=5

[Wed, 25 Feb 2026 06:58:50 GMT] 무주군, 난임부부 대상 한방 난임부부 치료 지원 - 무진장뉴스i
URL: https://news.google.com/rss/articles/CBMiWkFVX3lxTE9DdEdEeGJVdmZ3RGltS2Q4M0NEcWNVbFM4NUJOVV9HSkV6SUltNHEzeDVnWmNOZ2hKWk52elRyZU

In [4]:
def get_arxiv_papers(keywords, max_results=5):
    """
    arXiv API를 통해 관련 논문을 가져옵니다.
    """
    search = arxiv.Search(
        query=keywords,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )
    
    results = []
    for result in search.results():
        # 최근 7일 이내 논문인지 확인 (필요시 조절)
        results.append({
            "title": result.title,
            "link": result.entry_id,
            "published": result.published.strftime("%Y-%m-%d"),
            "summary": result.summary, # LLM 요약을 위해 원문 요약 포함
            "source": "arXiv"
        })
    return results

In [14]:
papers = get_arxiv_papers("Embryo quality", max_results=10)
for p in papers:
    print(f"[{p['published']}] {p['title']}\nURL: {p['link']}\n")

C:\Users\choiw\AppData\Local\Temp\ipykernel_35544\2888569095.py:12: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


[2026-02-27] Do LLMs Benefit From Their Own Words?
URL: http://arxiv.org/abs/2602.24287v1

[2026-02-27] Geometric Resilience of Quantum LiDAR in Turbulent Media: A Wasserstein Distance Approach
URL: http://arxiv.org/abs/2602.24280v1

[2026-02-27] Fermion Mass Hierarchy and a High Quality Axion From Gauged U(1) Flavor Symmetry
URL: http://arxiv.org/abs/2602.24253v1

[2026-02-27] Principal Component Analysis for ACS/WFC Superbias Temporal Variation
URL: http://arxiv.org/abs/2602.24244v1

[2026-02-27] Joint Geometric and Trajectory Consistency Learning for One-Step Real-World Super-Resolution
URL: http://arxiv.org/abs/2602.24240v1

[2026-02-27] Better Learning-Augmented Spanning Tree Algorithms via Metric Forest Completion
URL: http://arxiv.org/abs/2602.24232v1

[2026-02-27] SenCache: Accelerating Diffusion Model Inference via Sensitivity-Aware Caching
URL: http://arxiv.org/abs/2602.24208v1

[2026-02-27] Linear Polarization Variations and Circular Polarization are Common Among Airless Bod

In [1]:
import feedparser
import arxiv
from datetime import datetime, timedelta
import urllib.parse
from newspaper import Article, Config
from googlenewsdecoder import gnewsdecoder
import logging
import requests
import nltk
import xml.etree.ElementTree as ET

In [2]:
def resolve_google_news_url(url):
    """
    Google 뉴스 RSS URL을 디코딩하여 원본 기사 주소를 반환합니다.
    """
    try:
        # 1. googlenewsdecoder 시도
        result = gnewsdecoder(url)
        if result.get("status") and result.get("decoded_url"):
            return result["decoded_url"]
        
        # 2. 실패 시 requests로 리디렉션 추적 (일부 케이스 대응)
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
        }
        response = requests.get(url, headers=headers, allow_redirects=True, timeout=5)
        if response.url and "google.com" not in response.url:
            return response.url
            
        return url
    except Exception as e:
        print(f"URL 디코딩 실패: {e}")
        return url

In [3]:
def get_article_image(url):
    """
    newspaper4k 및 메타데이터를 사용하여 기사 URL에서 주요 이미지 URL을 추출합니다.
    """
    try:
        if "news.google.com" in url:
            url = resolve_google_news_url(url)
            
        if "google.com" in url and "rss/articles" not in url:
            return None

        article = Article(url, language='ko', config=config)
        article.download()
        article.parse()
        
        # 1. newspaper4k의 기본 top_image 시도
        image = article.top_image
        
        # 2. 실패 시 OpenGraph 또는 Twitter 메타데이터 직접 확인
        if not image or "googleusercontent.com" in image or "gstatic.com" in image:
            image = article.meta_data.get('og', {}).get('image')
            if not image:
                image = article.meta_data.get('twitter', {}).get('image')
        
        # 3. 절대 경로 확인 및 구글 서버 이미지 필터링
        if image:
            print("이미지 추출에 성공했습니다.")
            if not image.startswith('http'):
                # 상대 경로인 경우 기본 URL 결합 (간단한 처리)
                from urllib.parse import urljoin
                image = urljoin(url, image)
                
            if "googleusercontent.com" in image or "gstatic.com" in image:
                return None
            
        return image
    except Exception as e:
        print(f"이미지 추출 실패 ({url}): {e}")
        return None


In [4]:
# newspaper4k 설정 (타임아웃 등)
config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
config.request_timeout = 10

In [9]:
url = 'https://www.businesspost.co.kr/BP?command=article_view&num=431458'

In [10]:
article = Article(url, language='ko', config=config)
article.download()
article.parse()
        
# 1. newspaper4k의 기본 top_image 시도
image = article.top_image

In [11]:
image

'https://www.businesspost.co.kr/news/photo/202602/20260227144038_100953.jpg'

In [33]:
image_url = get_article_image(url)

이미지 추출에 성공했습니다.


In [34]:
image_url

'https://straitsresearch.com/uploads/reports/1715585083-fertility-services-market.jpg'

In [14]:
from crawler import get_google_news, get_pubmed_papers
import urllib.parse
import feedparser

In [21]:
news_items = get_google_news("난임 인공수정", days=7,max_results=10)

unable to cache publicsuffix.org-tlds.{'urls': ('https://publicsuffix.org/list/public_suffix_list.dat', 'https://raw.githubusercontent.com/publicsuffix/list/master/public_suffix_list.dat'), 'fallback_to_snapshot': True} in c:\ProgramData\anaconda3\envs\py31012\lib\site-packages\tldextract\.suffix_cache\publicsuffix.org-tlds\de84b5ca2167d4c83e38fb162f2e8738.tldextract.json. This could refresh the Public Suffix List over HTTP every app startup. Construct your `TLDExtract` with a writable `cache_dir` or set `cache_dir=None` to silence this warning. [WinError 5] 액세스가 거부되었습니다: 'c:\\ProgramData\\anaconda3\\envs\\py31012\\lib\\site-packages\\tldextract\\.suffix_cache'


이미지 추출 실패 (https://newsfield.net/%EB%8B%A8%EB%8F%85-%ED%95%9C%ED%99%94%EC%86%90%ED%95%B4%EB%B3%B4%ED%97%98-%EC%8B%9C%EA%B7%B8%EB%8B%88%EC%B2%98-%EC%97%AC%EC%84%B1%EB%B3%B4%ED%97%98-%EA%B3%A0%EB%A0%B9-%EC%82%B0%EB%AA%A8%C2%B7/): Article `download()` failed with HTTPSConnectionPool(host='newsfield.net', port=443): Read timed out. (read timeout=10) on URL https://newsfield.net/%EB%8B%A8%EB%8F%85-%ED%95%9C%ED%99%94%EC%86%90%ED%95%B4%EB%B3%B4%ED%97%98-%EC%8B%9C%EA%B7%B8%EB%8B%88%EC%B2%98-%EC%97%AC%EC%84%B1%EB%B3%B4%ED%97%98-%EA%B3%A0%EB%A0%B9-%EC%82%B0%EB%AA%A8%C2%B7/


In [22]:
keywords = "난임 인공수정"
days = 7

In [23]:
base_url = "https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko"
    # 최근 n일간의 검색 결과 필터링 (when:7d 방식)
query = f"{keywords} when:{days}d"
encoded_query = urllib.parse.quote(query)
rss_url = base_url.format(query=encoded_query)
    
feed = feedparser.parse(rss_url)
results = []

In [19]:
base_url = "https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko"
    # 최근 n일간의 검색 결과 필터링 (when:7d 방식)
query = f"{keywords} when:{days}d"
encoded_query = urllib.parse.quote(query)
rss_url = base_url.format(query=encoded_query)
    
feed = feedparser.parse(rss_url)
results = []

In [24]:
rss_url

'https://news.google.com/rss/search?q=%EB%82%9C%EC%9E%84%20%EC%9D%B8%EA%B3%B5%EC%88%98%EC%A0%95%20when%3A7d&hl=ko&gl=KR&ceid=KR:ko'

In [17]:
feed

{'bozo': False,
 'entries': [],
 'feed': {'generator_detail': {'name': 'NFE/5.0'},
  'generator': 'NFE/5.0',
  'title': '"난임 시험관 아기 인공수정 배아 수정란 when:7d" - Google 뉴스',
  'title_detail': {'type': 'text/plain',
   'language': None,
   'base': 'https://news.google.com/rss/search?q=%EB%82%9C%EC%9E%84%20%EC%8B%9C%ED%97%98%EA%B4%80%20%EC%95%84%EA%B8%B0%20%EC%9D%B8%EA%B3%B5%EC%88%98%EC%A0%95%20%EB%B0%B0%EC%95%84%20%EC%88%98%EC%A0%95%EB%9E%80%20when%3A7d&hl=ko&gl=KR&ceid=KR:ko',
   'value': '"난임 시험관 아기 인공수정 배아 수정란 when:7d" - Google 뉴스'},
  'links': [{'rel': 'alternate',
    'type': 'text/html',
    'href': 'https://news.google.com/search?q=%EB%82%9C%EC%9E%84+%EC%8B%9C%ED%97%98%EA%B4%80+%EC%95%84%EA%B8%B0+%EC%9D%B8%EA%B3%B5%EC%88%98%EC%A0%95+%EB%B0%B0%EC%95%84+%EC%88%98%EC%A0%95%EB%9E%80+when:7d&hl=ko&gl=KR&ceid=KR:ko'}],
  'link': 'https://news.google.com/search?q=%EB%82%9C%EC%9E%84+%EC%8B%9C%ED%97%98%EA%B4%80+%EC%95%84%EA%B8%B0+%EC%9D%B8%EA%B3%B5%EC%88%98%EC%A0%95+%EB%B0%B0%EC%95%84+%EC%88%98%